In [ ]:
# Install all required libraries
# Run this cell once at the beginning

!pip install -q transformers==4.40.0
!pip install -q torch torchvision
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q datasets
!pip install -q evaluate
!pip install -q rouge-score
!pip install -q bert-score
!pip install -q nltk
!pip install -q pandas numpy scikit-learn matplotlib seaborn
!pip install -q accelerate
!pip install -q tqdm

print("All libraries installed successfully!")

In [ ]:
# Core imports
import os
import re
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader

# Hugging Face
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    pipeline,
    TrainingArguments,
    Trainer
)
from sentence_transformers import SentenceTransformer

# Vector store
import faiss

# Evaluation
import nltk
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

# Utilities
from tqdm import tqdm

# Suppress non-critical warnings to keep output clean
warnings.filterwarnings('ignore')
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:

ticket_templates = {
    "billing": [
        "I was charged twice for my subscription this month. Please refund the duplicate charge.",
        "My invoice shows an incorrect amount. I should have been billed $29 not $59.",
        "I cancelled my subscription 3 weeks ago but I'm still being charged.",
        "I need a receipt for my last payment for tax purposes.",
        "The promo code SAVE20 didn't apply to my order. I want my discount.",
        "Why did my bill increase without any notification?",
        "I want to upgrade my plan but not sure what will be charged.",
        "I see an unauthorized charge on my account. This is fraud!",
    ],
    "technical": [
        "The app keeps crashing every time I try to open it on my iPhone.",
        "I can't log into my account. It says invalid password but I just reset it.",
        "The dashboard is loading very slowly. It takes more than 2 minutes.",
        "I'm getting error code 500 when I try to save my settings.",
        "The dark mode toggle isn't working on the website.",
        "My API keys stopped working after I rotated them yesterday.",
        "The mobile notification feature is not sending alerts.",
        "Integration with Slack is broken after your last update.",
    ],
    "shipping": [
        "My order was supposed to arrive 5 days ago. Where is it?",
        "The tracking number you sent me doesn't show any updates.",
        "I received the wrong item in my shipment.",
        "The package arrived damaged. I want a replacement sent immediately.",
        "Can I change the delivery address for my pending order?",
        "My order shows delivered but I never received it.",
        "I need express shipping for an urgent order. Is that available?",
        "The customs form is missing from my international shipment.",
    ],
    "account": [
        "I forgot my password and the reset email is not arriving.",
        "I need to update my email address associated with the account.",
        "How do I add another user to my team account?",
        "I want to delete my account and all my data permanently.",
        "My account was suspended and I don't know why.",
        "I can't change my username. The button is greyed out.",
        "I need to transfer my account to a different owner.",
        "Two-factor authentication is not working with my authenticator app.",
    ],
    "product": [
        "How do I export my data in CSV format?",
        "Does your product support integration with Salesforce?",
        "I'm looking for documentation on the advanced filtering feature.",
        "Can I use the product offline without an internet connection?",
        "What is the maximum file size I can upload?",
        "Is there a way to schedule automated reports?",
        "How many team members can I add on the Business plan?",
        "I need a feature to bulk import contacts. Does it exist?",
    ]
}

# Sarcastic versions of common complaints
sarcastic_tickets = [
    "Oh great, you charged me AGAIN. Because apparently one charge just isn't enough for you.",
    "Wow, amazing how the app crashes EVERY single time. Truly impressive engineering.",
    "Oh sure, my package is 'out for delivery' for the 5th day in a row. Totally normal.",
    "Fantastic! I reset my password 6 times and still can't log in. What a seamless experience.",
    "Your support team responded in only 3 days. Absolutely lightning fast service!",
]

# Reference responses for RAG evaluation
reference_responses = {
    "billing": "I sincerely apologize for the billing issue. I've reviewed your account and will process a full refund within 3-5 business days. You'll receive a confirmation email shortly.",
    "technical": "I'm sorry you're experiencing technical difficulties. Our engineering team has been notified. Please try clearing your cache or reinstalling the app. I'll follow up within 24 hours with an update.",
    "shipping": "I apologize for the delay with your shipment. I've escalated this with our logistics team and will provide a tracking update within 2 hours. If not resolved, we'll send a replacement.",
    "account": "I understand your concern with your account. I've flagged this for our account security team. Please check your spam folder for the reset email and contact us if you need further assistance.",
    "product": "Thank you for your question! I'm happy to help. Here's the information you requested along with links to our documentation. Please let me know if you have any follow-up questions.",
}


def build_dataset():
    """Build a labeled dataset from the ticket templates."""
    rows = []
    label_map = {cat: i for i, cat in enumerate(ticket_templates.keys())}

    for category, messages in ticket_templates.items():
        for msg in messages:
            rows.append({
                "text": msg,
                "category": category,
                "label": label_map[category],
                "sentiment": "negative" if any(w in msg.lower() for w in ["crashed", "wrong", "fraud", "broken", "damaged", "suspend"]) else "neutral",
                "is_sarcastic": 0,
                "reference_response": reference_responses[category]
            })

    # Add sarcastic tickets
    for msg in sarcastic_tickets:
        rows.append({
            "text": msg,
            "category": "technical",
            "label": 1,
            "sentiment": "negative",
            "is_sarcastic": 1,
            "reference_response": reference_responses["technical"]
        })

    return pd.DataFrame(rows)


df = build_dataset()
print(f"Dataset shape: {df.shape}")
print(f"\nCategory distribution:")
print(df['category'].value_counts())
print(f"\nSarcastic tickets: {df['is_sarcastic'].sum()}")
df.head()

In [ ]:
# Visualize the dataset distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Category distribution
df['category'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Ticket Category Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Sentiment distribution
df['sentiment'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                     colors=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[1].set_title('Sentiment Distribution', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

# Message length distribution
df['text_length'] = df['text'].apply(len)
axes[2].hist(df['text_length'], bins=20, color='mediumpurple', edgecolor='black')
axes[2].set_title('Message Length Distribution', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Character Count')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('dataset_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Dataset analysis chart saved as dataset_analysis.png")

In [ ]:
class TextPreprocessor:
    """
    Handles all text cleaning and normalization steps.
    We keep this as a class so it's reusable across different
    parts of the pipeline.
    """

    def __init__(self):
        # Common contractions to expand
        self.contractions = {
            "can't": "cannot", "won't": "will not", "don't": "do not",
            "I'm": "I am", "it's": "it is", "I've": "I have",
            "isn't": "is not", "wasn't": "was not", "didn't": "did not",
            "doesn't": "does not", "haven't": "have not", "couldn't": "could not"
        }

    def expand_contractions(self, text):
        for contraction, expansion in self.contractions.items():
            text = text.replace(contraction, expansion)
        return text

    def remove_special_chars(self, text, keep_punctuation=True):
        if keep_punctuation:
            # Keep letters, numbers, spaces, and basic punctuation
            text = re.sub(r'[^a-zA-Z0-9\s.,!?\'-]', ' ', text)
        else:
            text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
        return text

    def normalize_whitespace(self, text):
        return re.sub(r'\s+', ' ', text).strip()

    def preprocess(self, text, lowercase=True):
        """Full preprocessing pipeline."""
        text = self.expand_contractions(text)
        text = self.remove_special_chars(text)
        text = self.normalize_whitespace(text)
        if lowercase:
            text = text.lower()
        return text

    def preprocess_batch(self, texts):
        return [self.preprocess(t) for t in texts]


# Apply preprocessing
preprocessor = TextPreprocessor()
df['clean_text'] = preprocessor.preprocess_batch(df['text'].tolist())

# Show before/after comparison
print("=== Preprocessing Examples ===")
for i in range(3):
    print(f"\nOriginal: {df['text'].iloc[i]}")
    print(f"Cleaned:  {df['clean_text'].iloc[i]}")
    print("-" * 60)

In [ ]:
class TicketDataset(Dataset):
    """
    Custom PyTorch Dataset for ticket classification.
    Handles tokenization internally so the DataLoader stays clean.
    """

    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }


# Prepare train/test split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=SEED,
    stratify=df['label'].tolist()
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# Load BERT tokenizer
print("\nLoading BERT tokenizer...")
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Create dataset objects
train_dataset = TicketDataset(X_train, y_train, bert_tokenizer)
test_dataset = TicketDataset(X_test, y_test, bert_tokenizer)

print("Dataset objects created successfully.")
print(f"Sample encoding shape: {train_dataset[0]['input_ids'].shape}")

In [ ]:
# Fine-tune BERT for ticket classification
# We use Hugging Face Trainer API for clean training loops

NUM_LABELS = 5  # billing, technical, shipping, account, product
LABEL_NAMES = list(ticket_templates.keys())

print("Loading BERT model for sequence classification...")
bert_classifier = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=NUM_LABELS,
    id2label={i: name for i, name in enumerate(LABEL_NAMES)},
    label2id={name: i for i, name in enumerate(LABEL_NAMES)}
)

# Training configuration
training_args = TrainingArguments(
    output_dir='./bert_ticket_classifier',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=50,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=torch.cuda.is_available(),
    report_to='none'
)


def compute_metrics(eval_pred):
    """Compute F1 score during training evaluation."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)
    return {'f1': f1, 'accuracy': acc}


# Initialize Trainer
trainer = Trainer(
    model=bert_classifier,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Train the model
print("\nStarting BERT fine-tuning...")
trainer.train()
print("\nTraining complete!")

In [ ]:
print("Evaluating classifier on test set...")
predictions_output = trainer.predict(test_dataset)
y_pred = np.argmax(predictions_output.predictions, axis=-1)
y_true = y_test

# Print classification report
print("\n=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES))

# Plot confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.title('Confusion Matrix — Ticket Classification', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary metrics
f1 = f1_score(y_true, y_pred, average='weighted')
acc = accuracy_score(y_true, y_pred)
print(f"\nFinal Weighted F1-Score: {f1:.4f}")
print(f"Final Accuracy: {acc:.4f}")

In [ ]:
# Load pre-trained RoBERTa for sentiment analysis
print("Loading RoBERTa sentiment model...")

sentiment_pipeline = pipeline(
    task="text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment",
    device=0 if device == 'cuda' else -1,
    top_k=None  # replaces deprecated return_all_scores=True
)

SENTIMENT_LABELS = {0: 'negative', 1: 'neutral', 2: 'positive'}

print("Sentiment model loaded successfully.")


def analyze_sentiment(text):
    """
    Analyze sentiment of a text and return label + confidence.
    Returns the label with highest score.
    """
    results = sentiment_pipeline(text[:512])[0]  # returns list of dicts
    best = max(results, key=lambda x: x['score'])
    label_idx = int(best['label'].split('_')[1])
    return SENTIMENT_LABELS[label_idx], round(best['score'], 4)


# Test on sample tickets
print("\n=== Sentiment Analysis Examples ===")
test_samples = df.sample(5, random_state=SEED)
for _, row in test_samples.iterrows():
    label, conf = analyze_sentiment(row['text'])
    print(f"Text: {row['text'][:70]}...")
    print(f"  -> Sentiment: {label} (confidence: {conf})")
    print()

In [ ]:
# Emotion detection using a dedicated emotion classifier
print("Loading emotion detection model...")
emotion_pipeline = pipeline(
    task="text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    device=0 if device == 'cuda' else -1,
)
print("Emotion model loaded.")

def detect_emotions(text):
    """Return a sorted list of (emotion, confidence) pairs."""
    results = emotion_pipeline(text[:512], top_k=None)
    # results is either [[{...}]] or [{...}], normalize it
    if isinstance(results[0], list):
        results = results[0]
    return sorted(results, key=lambda x: x['score'], reverse=True)

# Visualize emotion scores for a few tickets
sample_tickets = [
    "I was charged twice and this is completely unacceptable!",
    "Thank you so much, my issue was resolved super quickly!",
    "I'm not sure what's happening with my order, can someone help?"
]

fig, axes = plt.subplots(1, len(sample_tickets), figsize=(18, 5))

for i, ticket in enumerate(sample_tickets):
    emotions = detect_emotions(ticket)
    labels = [e['label'] for e in emotions]
    scores = [e['score'] for e in emotions]
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(labels)))
    axes[i].barh(labels, scores, color=colors, edgecolor='grey')
    axes[i].set_title(f'Ticket {i+1}\n"{ticket[:45]}..."', fontsize=9)
    axes[i].set_xlabel('Confidence')
    axes[i].set_xlim(0, 1)

plt.suptitle('Emotion Detection Results', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('emotion_detection.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# For sarcasm detection we fine-tune a DistilBERT model
# on a labeled sarcasm dataset

from datasets import load_dataset

print("Loading sarcasm dataset...")
# Using the Headlines dataset — a well-known sarcasm benchmark
sarcasm_data = load_dataset("raquiba/Sarcasm_News_Headline", split='train')
sarcasm_df = sarcasm_data.to_pandas()

print(f"Sarcasm dataset size: {len(sarcasm_df)}")
print(f"Sarcastic examples: {sarcasm_df['is_sarcastic'].sum()}")
print(f"Non-sarcastic examples: {(sarcasm_df['is_sarcastic'] == 0).sum()}")
sarcasm_df.head()

In [ ]:
# Prepare sarcasm dataset and fine-tune DistilBERT
pos_samples = sarcasm_df[sarcasm_df['is_sarcastic'] == 1].sample(2000, random_state=SEED)
neg_samples = sarcasm_df[sarcasm_df['is_sarcastic'] == 0].sample(2000, random_state=SEED)
sarcasm_balanced = pd.concat([pos_samples, neg_samples]).sample(frac=1, random_state=SEED)

sarcasm_texts = sarcasm_balanced['headline'].tolist()
sarcasm_labels = sarcasm_balanced['is_sarcastic'].tolist()

# Split
sx_train, sx_test, sy_train, sy_test = train_test_split(
    sarcasm_texts, sarcasm_labels, test_size=0.2, random_state=SEED
)

# Load DistilBERT tokenizer
print("Loading DistilBERT tokenizer...")
sarcasm_tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

sarcasm_train_ds = TicketDataset(sx_train, sy_train, sarcasm_tokenizer)
sarcasm_test_ds  = TicketDataset(sx_test,  sy_test,  sarcasm_tokenizer)

# Load model
print("Loading DistilBERT for sarcasm detection...")
sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
)

# Fine-tune
sarcasm_args = TrainingArguments(
    output_dir='./sarcasm_model',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    eval_strategy='epoch',        # ← fixed (was evaluation_strategy)
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to='none'
)

sarcasm_trainer = Trainer(
    model=sarcasm_model,
    args=sarcasm_args,
    train_dataset=sarcasm_train_ds,
    eval_dataset=sarcasm_test_ds,
    compute_metrics=compute_metrics
)

print("\nFine-tuning sarcasm detector...")
sarcasm_trainer.train()
print("Training complete!")

# Evaluate
sarcasm_preds = np.argmax(sarcasm_trainer.predict(sarcasm_test_ds).predictions, axis=-1)
print(f"\nSarcasm Detection F1-Score: {f1_score(sy_test, sarcasm_preds, average='binary'):.4f}")
print(classification_report(sy_test, sarcasm_preds, target_names=['Not Sarcastic', 'Sarcastic']))

In [ ]:
# Build the knowledge base — these simulate your company's support docs

knowledge_base = [
    # Billing KB articles
    {"id": "KB001", "category": "billing", "title": "Refund Policy",
     "content": "We offer full refunds within 30 days of purchase. Refunds are processed within 3-5 business days to the original payment method. For subscription plans, prorated refunds are available for unused periods."},
    {"id": "KB002", "category": "billing", "title": "Billing Cycle",
     "content": "Subscriptions are billed monthly on the same date you first subscribed. Annual plans receive a 20% discount. Invoices are sent to your registered email 3 days before each billing date."},
    {"id": "KB003", "category": "billing", "title": "Payment Methods",
     "content": "We accept Visa, Mastercard, Amex, PayPal, and bank transfers for annual plans. Credit card details are encrypted using AES-256. You can update payment methods in Account Settings > Billing."},

    # Technical KB articles
    {"id": "KB004", "category": "technical", "title": "App Troubleshooting",
     "content": "If the app crashes, first try clearing cache from Settings > App Info > Clear Cache. If that doesn't work, uninstall and reinstall the latest version. Ensure your OS is updated to iOS 14+ or Android 10+."},
    {"id": "KB005", "category": "technical", "title": "Login Issues",
     "content": "Reset your password using the Forgot Password link. Check your spam folder for the reset email. If 2FA is enabled, make sure your authenticator app time is synced. Contact support if you still cannot log in."},
    {"id": "KB006", "category": "technical", "title": "API Documentation",
     "content": "API keys are valid for 90 days and can be rotated in the Developer Dashboard. After rotation, old keys remain active for 24 hours for zero-downtime migration. Rate limit is 1000 requests/minute on Business plan."},

    # Shipping KB articles
    {"id": "KB007", "category": "shipping", "title": "Shipping Policy",
     "content": "Standard shipping takes 3-7 business days. Express shipping (1-2 days) is available for an additional fee. International orders may take 10-21 days. Tracking numbers are emailed once your order ships."},
    {"id": "KB008", "category": "shipping", "title": "Damaged or Missing Items",
     "content": "For damaged items, please take photos and email them to support within 48 hours. We will ship a replacement within 2 business days at no cost. For missing packages, we investigate with the carrier before issuing a replacement."},

    # Account KB articles
    {"id": "KB009", "category": "account", "title": "Account Management",
     "content": "To update your email, go to Account Settings > Profile > Edit Email. A verification link will be sent to both old and new email addresses. Account deletion takes 30 days to process and cannot be undone."},
    {"id": "KB010", "category": "account", "title": "Team Management",
     "content": "Team admins can add users from Dashboard > Team > Invite Members. Business plan supports up to 25 members. Enterprise plan offers unlimited members with role-based access control."},

    # Product KB articles
    {"id": "KB011", "category": "product", "title": "Data Export",
     "content": "Data can be exported in CSV, JSON, or Excel format from Dashboard > Data > Export. Large exports (>100MB) are sent via email download link. Scheduled exports can be set up under Automation settings."},
    {"id": "KB012", "category": "product", "title": "Integrations",
     "content": "We support Salesforce, HubSpot, Slack, Zapier, and 50+ integrations via our API. Salesforce integration syncs contacts bi-directionally every 15 minutes. Setup guides are available in our integration docs."},
]

kb_df = pd.DataFrame(knowledge_base)
print(f"Knowledge base: {len(kb_df)} articles")
print(f"Categories covered: {kb_df['category'].unique().tolist()}")
kb_df.head()

In [ ]:
# Build the FAISS vector index for fast semantic search

print("Loading sentence transformer for embeddings...")
# all-MiniLM-L6-v2 is fast and produces good 384-dim embeddings
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all KB articles
print("Encoding knowledge base articles...")
kb_texts = [f"{doc['title']}. {doc['content']}" for doc in knowledge_base]
kb_embeddings = embedder.encode(kb_texts, show_progress_bar=True, convert_to_numpy=True)

# L2-normalize for cosine similarity search
faiss.normalize_L2(kb_embeddings)

# Build FAISS index (Inner Product = Cosine after L2 normalization)
EMBED_DIM = kb_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(EMBED_DIM)
faiss_index.add(kb_embeddings)

print(f"\nFAISS index built successfully.")
print(f"Embedding dimension: {EMBED_DIM}")
print(f"Total vectors indexed: {faiss_index.ntotal}")


def retrieve_context(query, top_k=3):
    """
    Given a user query, retrieve top-K most relevant KB articles.
    Returns the combined context text and individual article metadata.
    """
    query_emb = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_emb)

    scores, indices = faiss_index.search(query_emb, top_k)

    retrieved = []
    for score, idx in zip(scores[0], indices[0]):
        doc = knowledge_base[idx]
        retrieved.append({
            'id': doc['id'],
            'title': doc['title'],
            'content': doc['content'],
            'category': doc['category'],
            'similarity_score': round(float(score), 4)
        })

    context = "\n\n".join([f"[{d['id']}] {d['title']}: {d['content']}" for d in retrieved])
    return context, retrieved


# Test retrieval
print("\n=== Testing RAG Retrieval ===")
test_query = "I was charged twice for my subscription this month"
context, docs = retrieve_context(test_query)

print(f"Query: {test_query}")
print(f"\nTop retrieved documents:")
for doc in docs:
    print(f"  [{doc['id']}] {doc['title']} | Score: {doc['similarity_score']}")

In [ ]:
# Response generation using a T5-based model with RAG context
# We use flan-t5-base which is instruction-tuned and handles
# Q&A tasks well without further fine-tuning

print("Loading response generation model (flan-t5-base)...")
gen_tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')
gen_model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')
gen_model = gen_model.to(device)

print("Generation model loaded.")


def generate_response(ticket_text, context, max_length=200):
    """
    Generate a customer support response using the ticket text
    and the retrieved KB context.

    The prompt structure tells the model its role, provides context,
    then asks it to generate a professional response.
    """
    prompt = f"""You are a helpful customer support agent. Use the following knowledge base information to answer the customer's ticket professionally and concisely.

Knowledge Base:
{context}

Customer Ticket: {ticket_text}

Professional Response:"""

    inputs = gen_tokenizer(
        prompt,
        return_tensors='pt',
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens=max_length,
            num_beams=4,           # Beam search for better quality
            temperature=0.7,
            do_sample=False,
            early_stopping=True
        )

    response = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response


# Test the full RAG pipeline
print("\n=== Testing Full RAG Pipeline ===")
test_tickets = [
    "I was charged twice for my subscription. Please help.",
    "The app keeps crashing when I try to open settings.",
    "My package hasn't arrived after 8 days."
]

for ticket in test_tickets:
    context, retrieved_docs = retrieve_context(ticket)
    response = generate_response(ticket, context)
    print(f"Ticket: {ticket}")
    print(f"Retrieved: {', '.join([d['id'] for d in retrieved_docs])}")
    print(f"Response: {response}")
    print("-" * 70)

In [ ]:
# Load an NLI model to check factual consistency
# We check if the response is "entailed" by the KB context

print("Loading NLI model for hallucination detection...")
nli_pipeline = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if device == 'cuda' else -1
)
print("NLI model loaded.")


def detect_hallucination(response, context, threshold=0.5):
    """
    Check if the response is factually consistent with the context.

    Uses NLI entailment: if the response is entailed by the context,
    it's likely factually grounded. Otherwise, it may be a hallucination.

    Returns:
        is_hallucination (bool): True if response seems fabricated
        entailment_score (float): Confidence that response is supported by context
    """
    # Check entailment: context -> response
    result = nli_pipeline(
        sequences=context[:1000],  # Premise: KB context
        candidate_labels=[response[:200]],  # Hypothesis: generated response
        hypothesis_template="{}"
    )

    entailment_score = result['scores'][0]
    is_hallucination = entailment_score < threshold

    return is_hallucination, round(entailment_score, 4)


# Test hallucination detection
print("\n=== Hallucination Detection Tests ===")

test_cases = [
    {
        "context": "We offer full refunds within 30 days. Processing takes 3-5 business days.",
        "response": "I'll process your refund immediately. It will appear in your account within 3-5 business days.",
        "expected": "Grounded (No Hallucination)"
    },
    {
        "context": "We offer full refunds within 30 days. Processing takes 3-5 business days.",
        "response": "Your refund will appear instantly within 1 hour and we will also send you a $50 gift card.",
        "expected": "Hallucination (invented details)"
    }
]

for case in test_cases:
    is_hallucination, score = detect_hallucination(case['response'], case['context'])
    status = "⚠️ HALLUCINATION DETECTED" if is_hallucination else "✅ Response is Grounded"
    print(f"Response: {case['response'][:80]}...")
    print(f"Entailment Score: {score} | Status: {status}")
    print(f"Expected: {case['expected']}")
    print()

In [ ]:
# Evaluate response quality on a sample of tickets

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)


def evaluate_responses(tickets, references):
    """
    For each ticket, generate a response using RAG and compare
    against the reference response using BLEU, ROUGE, BERTScore.
    """
    generated_responses = []
    bleu_scores = []
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    print("Generating responses for evaluation...")
    for ticket in tqdm(tickets):
        context, _ = retrieve_context(ticket)
        response = generate_response(ticket, context)
        generated_responses.append(response)

        # ROUGE
        scores = scorer.score(response, references[tickets.index(ticket)])
        rouge1_scores.append(scores['rouge1'].fmeasure)
        rouge2_scores.append(scores['rouge2'].fmeasure)
        rougeL_scores.append(scores['rougeL'].fmeasure)

        # BLEU (sentence level)
        ref_tokens = [references[tickets.index(ticket)].split()]
        hyp_tokens = response.split()
        bleu = sentence_bleu(ref_tokens, hyp_tokens, weights=(0.5, 0.5))
        bleu_scores.append(bleu)

    # BERTScore (batch)
    print("\nComputing BERTScore...")
    P, R, F1 = bert_score_fn(
        generated_responses,
        references,
        lang='en',
        verbose=False
    )

    results = {
        'generated': generated_responses,
        'bleu': bleu_scores,
        'rouge1': rouge1_scores,
        'rouge2': rouge2_scores,
        'rougeL': rougeL_scores,
        'bert_precision': P.tolist(),
        'bert_recall': R.tolist(),
        'bert_f1': F1.tolist()
    }

    return results


# Build evaluation set
eval_tickets = [
    "I was charged twice for my subscription this month.",
    "The app keeps crashing on my iPhone.",
    "My order is delayed by 5 days.",
    "How do I export my data as a CSV?",
    "I forgot my password and cannot log in."
]

eval_references = [
    reference_responses["billing"],
    reference_responses["technical"],
    reference_responses["shipping"],
    reference_responses["product"],
    reference_responses["account"]
]

# Run evaluation
eval_results = evaluate_responses(eval_tickets, eval_references)

# Print summary
print("\n=== Evaluation Summary ===")
print(f"Average BLEU-2:        {np.mean(eval_results['bleu']):.4f}")
print(f"Average ROUGE-1:       {np.mean(eval_results['rouge1']):.4f}")
print(f"Average ROUGE-2:       {np.mean(eval_results['rouge2']):.4f}")
print(f"Average ROUGE-L:       {np.mean(eval_results['rougeL']):.4f}")
print(f"Average BERTScore F1:  {np.mean(eval_results['bert_f1']):.4f}")

In [ ]:
# Visualize evaluation metrics

metrics = ['BLEU-2', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BERTScore F1']
avg_scores = [
    np.mean(eval_results['bleu']),
    np.mean(eval_results['rouge1']),
    np.mean(eval_results['rouge2']),
    np.mean(eval_results['rougeL']),
    np.mean(eval_results['bert_f1'])
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of average scores
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
bars = axes[0].bar(metrics, avg_scores, color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('Average Evaluation Scores', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1)
for bar, score in zip(bars, avg_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{score:.3f}', ha='center', fontweight='bold')

# Per-ticket BERTScore
ticket_labels = [f"T{i+1}" for i in range(len(eval_tickets))]
axes[1].plot(ticket_labels, eval_results['bert_f1'], 'o-', color='#2196F3',
             linewidth=2, markersize=8, label='BERTScore F1')
axes[1].plot(ticket_labels, eval_results['rouge1'], 's--', color='#4CAF50',
             linewidth=2, markersize=8, label='ROUGE-1')
axes[1].set_title('Per-Ticket Evaluation Scores', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Score')
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Response Quality Evaluation', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
class SupportPipeline:
    """
    End-to-end customer support automation pipeline.

    Combines all NLP components into one unified interface:
    classification -> sentiment -> sarcasm -> RAG -> hallucination check
    """

    def __init__(
        self,
        classifier, classifier_tokenizer,
        sarcasm_clf, sarcasm_tokenizer,
        generator, gen_tokenizer,
        embedder, faiss_index, knowledge_base,
        preprocessor
    ):
        self.classifier = classifier
        self.classifier_tokenizer = classifier_tokenizer
        self.sarcasm_clf = sarcasm_clf
        self.sarcasm_tokenizer = sarcasm_tokenizer
        self.generator = generator
        self.gen_tokenizer = gen_tokenizer
        self.embedder = embedder
        self.faiss_index = faiss_index
        self.knowledge_base = knowledge_base
        self.preprocessor = preprocessor
        self.label_names = LABEL_NAMES

    def classify_ticket(self, text):
        """Predict ticket category using fine-tuned BERT."""
        inputs = self.classifier_tokenizer(
            text, return_tensors='pt', max_length=128,
            truncation=True, padding=True
        ).to(device)

        with torch.no_grad():
            logits = self.classifier(**inputs).logits

        probs = torch.softmax(logits, dim=-1).squeeze().cpu().tolist()
        predicted_idx = int(torch.argmax(logits))
        return self.label_names[predicted_idx], round(probs[predicted_idx], 4)

    def detect_sarcasm(self, text):
        """Detect sarcasm in a customer message."""
        inputs = self.sarcasm_tokenizer(
            text, return_tensors='pt', max_length=128,
            truncation=True, padding=True
        ).to(device)

        with torch.no_grad():
            logits = self.sarcasm_clf(**inputs).logits

        probs = torch.softmax(logits, dim=-1).squeeze().cpu().tolist()
        is_sarcastic = int(torch.argmax(logits)) == 1
        return is_sarcastic, round(probs[1], 4)

    def compute_priority(self, sentiment, is_sarcastic, category):
        """
        Compute a 1-3 priority score for routing and SLA.
        High priority tickets get routed to human agents.
        """
        score = 1  # Low priority by default
        if sentiment == 'negative':
            score += 1
        if is_sarcastic:
            score += 1  # Sarcasm often signals severe frustration
        if category in ['billing', 'account']:
            score = min(score + 1, 3)  # Billing/account issues are higher priority
        return min(score, 3)

    def process_ticket(self, ticket_text):
        """
        Full pipeline: analyze a customer ticket and generate a response.
        """
        # Step 1: Preprocess
        clean_text = self.preprocessor.preprocess(ticket_text)

        # Step 2: Classify
        category, cat_confidence = self.classify_ticket(clean_text)

        # Step 3: Analyze sentiment
        sentiment, sent_confidence = analyze_sentiment(ticket_text)

        # Step 4: Detect sarcasm
        is_sarcastic, sarcasm_score = self.detect_sarcasm(ticket_text)

        # Step 5: Retrieve relevant KB context
        context, retrieved_docs = retrieve_context(ticket_text)

        # Step 6: Generate response
        response = generate_response(ticket_text, context)

        # Step 7: Check for hallucinations
        is_hallucination, entailment_score = detect_hallucination(response, context)

        # Step 8: Compute priority
        priority = self.compute_priority(sentiment, is_sarcastic, category)

        return {
            'original_ticket': ticket_text,
            'category': category,
            'category_confidence': cat_confidence,
            'sentiment': sentiment,
            'sentiment_confidence': sent_confidence,
            'is_sarcastic': is_sarcastic,
            'sarcasm_score': sarcasm_score,
            'retrieved_docs': [d['id'] for d in retrieved_docs],
            'generated_response': response,
            'is_hallucination': is_hallucination,
            'entailment_score': entailment_score,
            'priority': priority
        }


# Instantiate the full pipeline
pipeline_obj = SupportPipeline(
    classifier=bert_classifier,
    classifier_tokenizer=bert_tokenizer,
    sarcasm_clf=sarcasm_model,
    sarcasm_tokenizer=sarcasm_tokenizer,
    generator=gen_model,
    gen_tokenizer=gen_tokenizer,
    embedder=embedder,
    faiss_index=faiss_index,
    knowledge_base=knowledge_base,
    preprocessor=preprocessor
)

print("Full pipeline initialized successfully!")